In [1]:
import pandas as pd
import numpy as np
import ast
import spacy 

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix
from pathlib import Path 

In [2]:
knn_encoded = pd.read_parquet("../data/processed/movies_with_keywords.parquet")

In [3]:
knn_encoded.head()

,id_film,type_titre,titre_principal,titre_original,titre_francais,genres,main_genre,duree_minutes,note_moyenne,nombre_votes,...,langue_originale,date_sortie,budget,recettes,patrimonial,zone_de_production,annee_de_sortie,genres_list,synopsis_fr,keywords_list
0,tt0035423,movie,Kate & Leopold,Kate & Leopold,Kate et Léopold,"[Comédie, Fantastique, Romance]",Comédie,118.0,6.4,93331,...,[anglais],2001-12-25,48000000.0,76019048.0,No,Américain,2001,"[Comédie, Fantastique, Romance]",Lorsque son ex-petit ami scientifique découvre...,"[new york city, time travel, duke, fish out of..."
1,tt0041640,movie,La Marie du Port,La Marie du Port,La Marie du port,"[Crime, Drame, Romance]",Crime,88.0,6.6,501,...,[français],1950-02-27,0.0,0.0,No,Français,1950,"[Crime, Drame, Romance]","Henri Chatelard a la quarantaine, possède un r...",[]
2,tt0041719,movie,Orpheus,Orphée,Orphée,"[Drame, Fantastique, Romance]",Drame,112.0,7.8,14394,...,[français],1950-09-29,0.0,0.0,No,Français,1950,"[Drame, Fantastique, Romance]",Un poète amoureux de la mort suit sa malheureu...,"[paris, france, mythology, poet, greek mytholo..."
3,tt0041931,movie,Stromboli,Stromboli (Terra di Dio),Stromboli,[Drame],Drame,81.0,7.2,8757,...,[italien],1950-02-15,900000.0,0.0,No,Américain,1950,[Drame],"Après la fin de la Seconde Guerre mondiale, un...","[fisherman, marriage of convenience, rural are..."
4,tt0042003,movie,A Man Walks in the City,Un homme marche dans la ville,Un homme marche dans la ville,[Drame],Drame,95.0,7.0,132,...,[français],1950-03-22,0.0,0.0,No,Français,1950,[Drame],"Le Havre, France, 1949. Dans une ville qui por...","[normandy, france, docker, le havre, france]"


In [4]:
import sys
print(sys.executable)

c:\Users\azerty\anaconda3\python.exe


In [5]:
print(repr(knn_encoded["keywords_list"].iloc[0]))
print(repr(knn_encoded["keywords_list"].iloc[1]))  # celle qui avait des données
print(repr(knn_encoded["keywords_list"].iloc[2]))

array(['new york city', 'time travel', 'duke', 'fish out of water',
       'falling in love', 'scientist', 'deadline', 'dog',
       'apartment building', 'tv commercial', 'brooklyn bridge',
       'ex-boyfriend ex-girlfriend relationship',
       'uncle nephew relationship', 'advertising executive',
       'broken elevator', 'jumping off a bridge', 'gentleman',
       'advertising agency', 'mental hospital', 'time portal',
       'brother sister relationship', '2000s', '1870s', 'propriety',
       'marry for money', 'big sister little brother',
       'scientific theory'], dtype=object)
array([], dtype=object)
array(['paris, france', 'mythology', 'poet', 'greek mythology',
       'romantic poet', 'orpheus'], dtype=object)


In [6]:
knn_encoded.dtypes

id_film                        object
type_titre                     object
titre_principal                object
titre_original                 object
titre_francais                 object
genres                         object
main_genre                     object
duree_minutes                 float64
note_moyenne                  float64
nombre_votes                    int64
decennie                        int64
noms_realisateurs              object
principaux_acteurs             object
synopsis                       object
slogan                         object
url_affiche                    object
url_fond                       object
societes_production            object
pays_production                object
langues_parlees                object
langue_originale               object
date_sortie            datetime64[ns]
budget                        float64
recettes                      float64
patrimonial                    object
zone_de_production             object
annee_de_sor

In [7]:
# =========================
# 1) NETTOYAGE MINIMUM
# =========================
needed = [
    "titre_francais", "genres_list", "zone_de_production", "decennie",
    "langue_originale", "nombre_votes", "synopsis"
]
missing = [c for c in needed if c not in knn_encoded.columns]
if missing:
    raise ValueError(f"Colonnes manquantes: {missing}")

knn_encoded = knn_encoded.dropna(subset=["titre_francais", "genres", "nombre_votes", "decennie"]).copy()

knn_encoded["nombre_votes"] = pd.to_numeric(knn_encoded["nombre_votes"], errors="coerce")
knn_encoded["decennie"] = pd.to_numeric(knn_encoded["decennie"], errors="coerce")
knn_encoded = knn_encoded.dropna(subset=["nombre_votes", "decennie"]).copy()

knn_encoded["nombre_votes"] = knn_encoded["nombre_votes"].astype(int)
knn_encoded["decennie"] = knn_encoded["decennie"].astype(int)

knn_encoded["synopsis_clean"] = knn_encoded["synopsis"].fillna("").astype(str)

print("Après nettoyage :", knn_encoded.shape)



Après nettoyage : (10307, 31)


In [8]:
#AFFICHER LES COLONNES NUMPY ARRAY
for col in ["zone_de_production", "langue_originale", "decennie"]:
    types = knn_encoded[col].apply(type).value_counts()
    print(f"\n{col}:\n{types}")

    mask = knn_encoded[col].apply(lambda x: isinstance(x, np.ndarray))
    if mask.any():
        print(f"  -> {mask.sum()} numpy arrays trouvés !")
        print(knn_encoded[col][mask].head())


zone_de_production:
zone_de_production
<class 'str'>    10307
Name: count, dtype: int64

langue_originale:
langue_originale
<class 'numpy.ndarray'>    10307
Name: count, dtype: int64
  -> 10307 numpy arrays trouvés !
0     [anglais]
1    [français]
2    [français]
3     [italien]
4    [français]
Name: langue_originale, dtype: object

decennie:
decennie
<class 'int'>    10307
Name: count, dtype: int64


In [9]:
# 2) NETTOYAGE ET PARSE

def parse_pylist(val):
    if isinstance(val, np.ndarray):
        return val.tolist()
    if isinstance(val, list):
        return val
    return []

# Parser les keywords
knn_encoded["keywords_list"] = knn_encoded["keywords_list"].apply(parse_pylist)
print(knn_encoded["keywords_list"].head())
print(type(knn_encoded["keywords_list"].iloc[0]))
print(knn_encoded["keywords_list"].iloc[0])

0    [new york city, time travel, duke, fish out of...
1                                                   []
2    [paris, france, mythology, poet, greek mytholo...
3    [fisherman, marriage of convenience, rural are...
4         [normandy, france, docker, le havre, france]
Name: keywords_list, dtype: object
<class 'list'>
['new york city', 'time travel', 'duke', 'fish out of water', 'falling in love', 'scientist', 'deadline', 'dog', 'apartment building', 'tv commercial', 'brooklyn bridge', 'ex-boyfriend ex-girlfriend relationship', 'uncle nephew relationship', 'advertising executive', 'broken elevator', 'jumping off a bridge', 'gentleman', 'advertising agency', 'mental hospital', 'time portal', 'brother sister relationship', '2000s', '1870s', 'propriety', 'marry for money', 'big sister little brother', 'scientific theory']


In [10]:
# =========================
# 2) GENRES
# =========================
# on garde seulement ceux qui ont au moins 1 genre
knn_encoded = knn_encoded[knn_encoded["genres_list"].apply(len) > 0].copy()
knn_encoded = knn_encoded.reset_index(drop=True)

knn_encoded[["titre_francais", "genres_list"]].head()


,titre_francais,genres_list
0,Kate et Léopold,"[Comédie, Fantastique, Romance]"
1,La Marie du port,"[Crime, Drame, Romance]"
2,Orphée,"[Drame, Fantastique, Romance]"
3,Stromboli,[Drame]
4,Un homme marche dans la ville,[Drame]


In [11]:
#2) GENRES PARSE

knn_encoded["genres_list"] = knn_encoded["genres_list"].apply(parse_pylist)
print(type(knn_encoded["genres_list"].iloc[0]))

<class 'list'>


In [13]:
#2) LANGUE ORIGINALE le premier élément du numpy array
knn_encoded["langue_originale"] = knn_encoded["langue_originale"].apply(
    lambda x: x[0] if isinstance(x, np.ndarray) and len(x) > 0 else (x if isinstance(x, str) else None)
)

# Vérification
print(type(knn_encoded["langue_originale"].iloc[0]))  # <class 'str'>
print(knn_encoded["langue_originale"].head())

<class 'str'>
0     anglais
1    français
2    français
3     italien
4    français
Name: langue_originale, dtype: object


In [59]:
# Genres incompatibles : si le film source a ce genre principal,
# exclure les recos qui ont un de ces genres comme genre principal
GENRE_EXCLUSIONS = {
    "Action": ["Comédie", "Romance", "Famille", "Fantastique"],
    "Crime" : ["Comédie","Famille"],
    "Drame": ["Animation", "Famille"],
    "Thriller": ["Comédie", "Famille"],
    "Horreur": ["Animation", "Comédie", "Romance", "Famille"],
    "Science-fiction": ["Comédie", "Romance", "Famille"],
    "Comédie": ["Horreur", "Guerre"],
    "Animation": ["Horreur", "Guerre", "Crime"],
    "Romance": ["Horreur",],
    "Famille": ["Horreur", "Guerre", "Crime", "Thriller"],
}

In [15]:
# =========================
# 3) VOTES -> CONTINU (log1p + standardisation)
# ecplication : distribution très skewed (importante asymétrie au niveau du nombre de votes, on compresse les très grandes valeurs avec log)
# => rend la distribution plus normale
# =========================

votes_log = np.log1p(knn_encoded["nombre_votes"].values.reshape(-1, 1))

scaler_votes = StandardScaler(with_mean=False)
X_votes = csr_matrix(scaler_votes.fit_transform(votes_log))

In [60]:
# =========================
# 4) ENCODAGE FEATURES -> X_final
# =========================

# A) Genres -> MultiLabelBinarizer
mlb = MultiLabelBinarizer(sparse_output=True)
X_genres = mlb.fit_transform(knn_encoded["genres_list"]).tocsr()

# A2) Keywords -> TF-IDF (pas MLB, pour capturer la rareté des mots-clés)
knn_encoded["keywords_clean"] = knn_encoded["keywords_list"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)
tfidf_kw = TfidfVectorizer(
    min_df=3,
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    dtype=np.float32
)
X_keywords = tfidf_kw.fit_transform(knn_encoded["keywords_clean"])
print("Keywords TF-IDF shape:", X_keywords.shape)

# B) Zone / Langue -> one-hot
X_zone = csr_matrix(
    pd.get_dummies(knn_encoded["zone_de_production"], prefix="zone", dummy_na=True)
    .astype("int8")
    .values
)

X_lang = csr_matrix(
    pd.get_dummies(knn_encoded["langue_originale"], prefix="lang", dummy_na=True)
    .astype("int8")
    .values
)

# C) Décennie -> one-hot
X_dec = csr_matrix(
    pd.get_dummies(knn_encoded["decennie"], prefix="dec", dummy_na=True)
    .astype("int8")
    .values
)

# D) Synopsis -> TF-IDF avec lemmatisation spaCy
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

knn_encoded["synopsis_clean"] = (
    knn_encoded["synopsis_clean"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

knn_encoded["synopsis_clean"] = [
    " ".join([token.lemma_ for token in doc
              if not token.is_stop
              and not token.is_punct
              and not token.is_space])
    for doc in nlp.pipe(knn_encoded["synopsis_clean"], batch_size=500)
]

tfidf = TfidfVectorizer(
    min_df=5,
    max_features=8000,
    ngram_range=(1, 1),
    sublinear_tf=True,
    dtype=np.float32
)

X_syn = tfidf.fit_transform(knn_encoded["synopsis_clean"])
print("Synopsis TF-IDF shape:", X_syn.shape)

Keywords TF-IDF shape: (10307, 5000)
Synopsis TF-IDF shape: (10307, 6517)


In [17]:
knn_encoded["synopsis_clean"].iloc[0]


'scientist ex boyfriend discover portal travel time bring 19th century nobleman name leopold prove skeptical kate reluctantly take responsibility show leopold 21st century time kate spend leopold hard fall doesn t return time absence forever alter history'

In [61]:
# =========================
# 5) PONDÉRATIONS (cinéma rural FR / seniors)
# =========================

W_SYN      = 0.30
W_KEYWORDS = 0.45
W_GENRES   = 0.086
W_DEC      = 0.08
W_LANG     = 0.06
W_ZONE     = 0.06
W_VOTES    = 0.02

# =========================
# Construction de X_final
# =========================

X_final = hstack([
    X_syn.multiply(W_SYN),
    X_keywords.multiply(W_KEYWORDS),
    X_genres.multiply(W_GENRES),
    X_dec.multiply(W_DEC),
    X_lang.multiply(W_LANG),
    X_zone.multiply(W_ZONE),
    X_votes.multiply(W_VOTES),
]).tocsr()

print("X_final shape:", X_final.shape)

X_final shape: (10307, 11639)


In [19]:
from scipy.sparse import save_npz
import os

save_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "X_final.npz")

save_npz(save_path, X_final)
print(f"Sauvegardé: {save_path}")

Sauvegardé: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed\X_final.npz


In [20]:
knn_encoded["synopsis_clean"].head()

0    scientist ex boyfriend discover portal travel ...
1    henri chatelard forty own restaurant cinema ci...
2       poet love death follow unhappy wife underworld
3    end wwii young lithuanian woman young italian ...
4    le havre france 1949 town show scar war friend...
Name: synopsis_clean, dtype: object

In [21]:
knn_encoded.to_parquet(r"C:\Users\azerty\Desktop\Projet2 - Copie\data\processed\knn_df_encoded.parquet", index=False)

In [22]:
print("synopsis_clean" in knn_encoded.columns)

True


In [74]:
# =========================
# 5) KNN + RECOMMANDATION (avec re-ranking popularité)
# =========================
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

OVERSAMPLE = 80
knn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=OVERSAMPLE)
knn_model.fit(X_final)

_pop_global_log = np.log1p(knn_encoded["nombre_votes"].fillna(0).astype(float).values)
_pop_global_max = _pop_global_log.max() 

def recommend_movies(title, k=6, year=None, oversample=OVERSAMPLE, alpha_pop=0.25):
    """
    1) Récupère des voisins par similarité cosine (contenu)
    2) Re-rank ces voisins avec un bonus de popularité (nombre_votes)

    alpha_pop: poids du bonus popularité (0.15 à 0.35 recommandé)
    oversample: nb de voisins candidats récupérés avant re-ranking 
    """

    # 1) trouver le film
    s = knn_encoded["titre_francais"].fillna("").astype(str).str.lower().str.strip()
    candidates = knn_encoded[s == str(title).lower().strip()]
    if candidates.empty:
        raise ValueError(f"Film introuvable : {title}")

    # 2) choisir l'index (si doublons et année fournie)
    if year is not None:
        cand_year = candidates[candidates["annee_de_sortie"] == int(year)]
        if cand_year.empty:
            years = sorted(candidates["annee_de_sortie"].dropna().unique().tolist())
            raise ValueError(f"'{title}' existe mais pas en {year}. Années dispo: {years}")
        candidates = cand_year

    idx = candidates.index[0]

    # 3) voisins (sur-échantillonnage)
    n = min(oversample, X_final.shape[0])
    distances, indices = knn_model.kneighbors(X_final[idx], n_neighbors=n)
    distances = distances.ravel()
    indices = indices.ravel()

    # 4) exclure le film lui-même
    mask = indices != idx
    distances = distances[mask]
    indices = indices[mask]

    if len(indices) == 0:
        return pd.DataFrame({"message": ["Aucune recommandation trouvée."]})


    # 5) re-ranking popularité sur les candidats
    # popularité = log1p(nombre_votes) normalisée
    if "nombre_votes" in knn_encoded.columns and alpha_pop > 0:
        pop_norm = _pop_global_log[indices] / (_pop_global_max + 1e-9)

        sim = 1 - distances
        sim_norm = (sim - sim.min()) / (sim.max() - sim.min() + 1e-9)

        final = (1 - alpha_pop) * sim_norm + alpha_pop * pop_norm
        order = np.argsort(final)[::-1]          # meilleur score d'abord

        indices = indices[order]
        distances = distances[order]
    
    # Filtrer genres incompatibles
    source_main_genre = knn_encoded.loc[idx, "main_genre"]
    excluded = GENRE_EXCLUSIONS.get(source_main_genre, [])
    
    if excluded:
        excluded_set = set(excluded)
        rec_genres = knn_encoded.loc[indices, "genres_list"]
        mask_compatible = rec_genres.apply(
            lambda g: len(set(g).intersection(excluded_set)) == 0 if isinstance(g, list) else True
        )
        indices = indices[mask_compatible.values]
        distances = distances[mask_compatible.values]

    # 6) garder k recos
    rec_idx = indices[:k].tolist()
    rec_dist = distances[:k].tolist()

    cols = ["titre_francais", "annee_de_sortie", "genres",
            "zone_de_production", "langue_originale", "votes_interval", "nombre_votes"]
    cols = [c for c in cols if c in knn_encoded.columns]

    out = knn_encoded.loc[rec_idx, cols].copy()
    out["distance_cosine"] = rec_dist

    # IMPORTANT: on ne trie plus par distance, car on a déjà re-ranké
    return out.reset_index(drop=True)

# Test
recommend_movies("Intouchables", k=6, alpha_pop=0.26)



,titre_francais,annee_de_sortie,genres,zone_de_production,langue_originale,nombre_votes,distance_cosine
0,Les apprentis,1995,"[Comédie, Drame]",Français,français,1255,0.576605
1,Célibataires... ou presque,2014,"[Comédie, Romance]",Américain,anglais,110079,0.661443
2,La Tête à l'envers,2017,"[Comédie, Crime, Drame]",Européen,allemand,2985,0.644857
3,The Climb,2019,"[Comédie, Drame]",Américain,anglais,5509,0.648330
4,This Crazy Heart,2017,"[Comédie, Drame]",Européen,allemand,4126,0.683740
5,Dialogue avec mon jardinier,2007,"[Comédie, Drame]",Français,français,4013,0.687022


In [73]:
recommend_movies("Le Parrain", k=6, alpha_pop=0.26)


,titre_francais,annee_de_sortie,genres,zone_de_production,langue_originale,nombre_votes,distance_cosine
0,"Le Parrain, 2ᵉ partie",1974,"[Crime, Drame]",Américain,anglais,1477725,0.614975
1,Les Affranchis,1990,"[Biographie, Crime, Drame]",Américain,anglais,1374264,0.746693
2,Les sentiers de la perdition,2002,"[Crime, Drame, Thriller]",Américain,anglais,298745,0.740953
3,Belly,1998,"[Crime, Drame]",Américain,anglais,13637,0.729499
4,Scarface,1983,"[Crime, Drame, Action]",Américain,anglais,1002394,0.748907
5,Manon des sources,1986,[Drame],Français,français,25489,0.733135


In [75]:
recommend_movies("Scarface", k=10, alpha_pop=0.26)


,titre_francais,annee_de_sortie,genres,zone_de_production,langue_originale,nombre_votes,distance_cosine
0,Un homme à part,2003,"[Action, Crime, Drame]",Américain,anglais,50999,0.649677
1,Gangster Number One,2000,"[Crime, Drame, Thriller, Action]",Européen,anglais,15718,0.686167
2,Le marginal,1983,"[Action, Crime, Drame, Thriller]",Français,français,3909,0.720851
3,Le Parrain,1972,"[Crime, Drame]",Américain,anglais,2198185,0.748907
4,Shotgun Stories,2007,"[Drame, Thriller]",Américain,anglais,11928,0.741531
5,Casino,1995,"[Crime, Drame, Histoire]",Français,anglais,611926,0.756711
6,Braveheart,1995,"[Biographie, Drame, Guerre, Action, Histoire]",Américain,anglais,1152486,0.759383
7,London Boulevard,2010,"[Crime, Drame]",Américain,anglais,51603,0.749205
8,Heli,2013,"[Crime, Drame, Romance]",Français,espagnol,5622,0.745431
9,Les ensorcelés,1952,"[Drame, Romance]",Américain,anglais,17633,0.753500


In [76]:
recommend_movies("Inception", k=10, alpha_pop=0.26)


,titre_francais,annee_de_sortie,genres,zone_de_production,langue_originale,nombre_votes,distance_cosine
0,Matrix,1999,"[Action, Science-fiction, Science-fiction]",Américain,anglais,2222591,0.766147
1,Total Recall : Mémoires programmées,2012,"[Action, Aventure, Science-fiction, Science-fi...",Américain,anglais,275334,0.801603
2,Interstellar,2014,"[Aventure, Drame, Science-fiction, Science-fic...",Américain,anglais,2469178,0.816468
3,Punch 119,2013,"[Action, Crime, Thriller, Aventure]",Américain,anglais,35556,0.804369
4,Le piège,1973,"[Thriller, Drame]",Européen,anglais,5738,0.801102
5,Total Recall,1990,"[Action, Aventure, Science-fiction, Science-fi...",Américain,anglais,377900,0.817753
6,Mission: Impossible,1996,"[Action, Aventure, Thriller]",Américain,anglais,513143,0.819641
7,Dune : Deuxième Partie,2023,"[Action, Aventure, Drame, Science-fiction]",Américain,anglais,711622,0.829182
8,Criminal: Un espion dans la tête,2016,"[Action, Science-fiction, Thriller, Crime, Sci...",Français,anglais,74407,0.823024
9,Le diabolique docteur Mabuse,1960,"[Crime, Mystère, Thriller]",Français,allemand,4213,0.813763


In [78]:
recommend_movies("Titanic", k=10, year = 1997, alpha_pop=0.26)

,titre_francais,annee_de_sortie,genres,zone_de_production,langue_originale,nombre_votes,distance_cosine
0,La légende du pianiste sur l'océan,1998,"[Drame, Musique, Romance]",Européen,italien,73088,0.651918
1,Love Story,1970,"[Drame, Romance]",Américain,anglais,39859,0.735086
2,L'aventure du Poséidon,1972,"[Action, Aventure, Drame, Thriller]",Américain,anglais,53363,0.742142
3,Les yeux noirs,1987,"[Comédie, Drame, Romance]",Européen,russe,3792,0.734610
4,USS Greyhound : La Bataille de l'Atlantique,2020,"[Drame, Histoire, Guerre, Action]",Américain,anglais,130738,0.776298
5,Un amour sans fin,2014,"[Drame, Mystère, Romance]",Américain,anglais,49043,0.772951
6,The Impossible,2012,"[Drame, Histoire, Thriller, Aventure]",Américain,anglais,261112,0.780598
7,Upside Down,2012,"[Drame, Romance, Science-fiction, Science-fict...",Français,anglais,75596,0.775993
8,Roméo & Juliette,1996,"[Drame, Romance]",Américain,anglais,257887,0.783008
9,Coeur sauvage,1993,"[Drame, Romance, Comédie]",Américain,anglais,16879,0.776604


In [28]:
print(f"X_final shape: {X_final.shape}")
print(f"X_syn shape: {X_syn.shape}")
print(f"X_keywords shape: {X_keywords.shape}")
print(f"X_genres shape: {X_genres.shape}")


X_final shape: (10307, 11726)
X_syn shape: (10307, 6604)
X_keywords shape: (10307, 5000)
X_genres shape: (10307, 20)


In [29]:
print(knn_encoded[knn_encoded["titre_francais"] == "Inception"]["main_genre"].values)
print(knn_encoded[knn_encoded["titre_francais"] == "Mary à tout prix"]["genres_list"].values)
print(knn_encoded[knn_encoded["titre_francais"] == "Dreams"]["genres_list"].values)

['Action']
[list(['Comédie', 'Romance'])]
[list(['Aventure', 'Animation', 'Comédie', 'Famille'])]


In [30]:
knn_cosine = NearestNeighbors(n_neighbors=11, metric="cosine")
knn_euclidean = NearestNeighbors(n_neighbors=11, metric="euclidean")

knn_cosine.fit(X_final)
knn_euclidean.fit(X_final)

,n_neighbors,11
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'euclidean'
,p,2
,metric_params,None
,n_jobs,None


In [31]:
#QUEL METRIC ON VA UTILISER ?

def recommend_with_model(title, model, k=10, year=None, max_year_gap=None, year_penalty=0.0):
    """
    - title : titre_francais (match exact, insensible à la casse)
    - model : knn_cosine ou knn_euclidean
    - k : nb de reco
    - year : optionnel, pour choisir le bon film si doublons
    - max_year_gap : optionnel, filtre dur (ex: 20 => garde +/- 20 ans)
    - year_penalty : optionnel, pénalité douce ajoutée à la distance (ex: 0.15)
                    score = distance + year_penalty * (|diff_year|/10)
    """

    # 1) Trouver le film
    candidates = knn_encoded[
        knn_encoded["titre_francais"].fillna("").str.lower().str.strip()
        == title.lower().strip()
    ]

    if candidates.empty:
        raise ValueError(f"Film introuvable : {title}")

    # 2) Désambiguïser avec l'année si fournie
    if year is not None:
        candidates_year = candidates[candidates["annee_de_sortie"] == year]
        if not candidates_year.empty:
            candidates = candidates_year

    idx = candidates.index[0]
    base_year = int(knn_encoded.loc[idx, "annee_de_sortie"])

    # 3) Récupérer un peu plus de voisins pour pouvoir filtrer après
    distances, indices = model.kneighbors(X_final[idx], n_neighbors=k + 1 + 50)
    distances = distances.flatten()
    indices = indices.flatten()

    results = knn_encoded.iloc[indices].copy()
    results["distance"] = distances

    # 4) Enlever le film lui-même
    results = results[results.index != idx].copy()

    # 5) Option : filtre dur sur l'écart d'année
    results["year_gap"] = (results["annee_de_sortie"].astype(int) - base_year).abs()
    if max_year_gap is not None:
        results = results[results["year_gap"] <= max_year_gap]

    # 6) Option : pénalité douce (reclassement)
    if year_penalty and year_penalty > 0:
        results["score"] = results["distance"] + year_penalty * (results["year_gap"] / 10.0)
        results = results.sort_values("score", ascending=True)
    else:
        results = results.sort_values("distance", ascending=True)

    # 7) Colonnes d'affichage
    cols = [c for c in [
        "titre_francais", "annee_de_sortie", "genres_list",
        "zone_de_production", "langue_originale", "votes_interval",
        "distance", "year_gap"
    ] if c in results.columns]

    return results[cols].head(k).reset_index(drop=True)


# ------------------------------------------------------------
# Comparer pour un film donné
# ------------------------------------------------------------
film_test = "Le Parrain"  # mets un titre qui existe dans ton "titre_francais"

print("🎬 Film testé :", film_test)

print("\n🔵 Recommandations - Distance COSINE")
rec_cosine = recommend_with_model(
    film_test, knn_cosine, k=10,
    max_year_gap=25,      # optionnel : filtre dur
    year_penalty=0.10     # optionnel : pénalité douce (mets 0 pour désactiver)
)
display(rec_cosine)

print("\n🟢 Recommandations - Distance EUCLIDIENNE")
rec_euclid = recommend_with_model(
    film_test, knn_euclidean, k=10,
    max_year_gap=25,
    year_penalty=0.10
)
display(rec_euclid)


# ------------------------------------------------------------
# Comparer les titres recommandés
# ------------------------------------------------------------
set_cosine = set(rec_cosine["titre_francais"])
set_euclid = set(rec_euclid["titre_francais"])

common = set_cosine.intersection(set_euclid)

print("\n📊 Analyse comparative :")
print("Films communs :", len(common))
print("Films différents :", len(set_cosine.symmetric_difference(set_euclid)))

print("\nFilms communs :")
print(common)


🎬 Film testé : Le Parrain

🔵 Recommandations - Distance COSINE


,titre_francais,annee_de_sortie,genres_list,zone_de_production,langue_originale,distance,year_gap
0,"Le Parrain, 2ᵉ partie",1974,"[Crime, Drame]",Américain,anglais,0.615014,2
1,Passeport pour deux tueurs,1972,"[Action, Crime, Thriller, Drame]",Européen,italien,0.740796,0
2,Echec à l'organisation,1973,"[Crime, Drame, Thriller]",Américain,anglais,0.733897,1
3,Mr. Majestic,1974,"[Action, Crime, Drame, Thriller]",Américain,anglais,0.736638,2
4,Jeremiah Johnson,1972,"[Aventure, Drame, Western, Histoire]",Américain,anglais,0.758736,0
5,Borsalino and Co.,1974,"[Action, Crime, Drame]",Français,français,0.746931,2
6,Joe Kidd,1972,"[Drame, Western]",Américain,anglais,0.767001,0
7,Big Guns - Les grands fusils,1973,"[Action, Crime, Drame]",Français,italien,0.757066,1
8,Nos plus belles années,1973,"[Drame, Romance]",Américain,anglais,0.767837,1
9,La mafia fait la loi,1968,"[Crime, Drame, Mystère]",Français,italien,0.747081,4



🟢 Recommandations - Distance EUCLIDIENNE


,titre_francais,annee_de_sortie,genres_list,zone_de_production,langue_originale,distance,year_gap
0,Société anonyme anti-crime,1972,"[Crime, Drame]",Français,italien,0.629254,0
1,Police puissance 7,1973,"[Action, Crime, Drame, Thriller]",Américain,anglais,0.632854,1
2,Les Gentilshommes de la chance,1971,"[Comédie, Crime, Drame]",Européen,russe,0.633200,1
3,Juste avant la nuit,1971,"[Crime, Drame]",Français,français,0.633638,1
4,Le témoin à abattre,1973,"[Crime, Drame, Thriller]",Français,italien,0.634109,1
5,Chers Parents,1973,[Drame],Français,italien,0.636049,1
6,Le boss,1973,"[Action, Crime, Thriller, Drame]",Européen,italien,0.638344,1
7,Comptes à rebours,1970,"[Crime, Drame]",Français,français,0.629701,2
8,"Vincent, François, Paul et les autres...",1974,[Drame],Français,français,0.635997,2
9,La horse,1970,"[Crime, Thriller, Drame]",Français,français,0.638010,2



📊 Analyse comparative :
Films communs : 0
Films différents : 20

Films communs :
set()


In [32]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

# racine projet
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent

PROJECT_ROOT = p
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
os.makedirs(DATA_PROCESSED, exist_ok=True)

print("DATA_RAW:", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)

DATA_RAW: c:\Users\azerty\Desktop\Projet2 - Copie\data\raw
DATA_PROCESSED: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed


In [33]:
out = DATA_PROCESSED / "knn_df_encoded.csv"
knn_encoded.to_csv(out, index=False)
print("Saved:", out)

Saved: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed\knn_df_encoded.csv


In [34]:
out = DATA_PROCESSED / "knn_df_encoded.parquet"
knn_encoded.to_parquet(out, index=False)
print("Saved:", out)

Saved: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed\knn_df_encoded.parquet


In [35]:
knn = pd.read_parquet(r"C:\Users\azerty\Desktop\Projet2 - Copie\data\processed\knn_df_encoded.parquet")
print(len(knn))
print(knn[knn["id_film"] == "tt1375666"])

10307
        id_film type_titre titre_principal titre_original titre_francais  \
6878  tt1375666      movie       Inception      Inception      Inception   

                                                 genres main_genre  \
6878  [Action, Aventure, Science-fiction, Science-fi...     Action   

      duree_minutes  note_moyenne  nombre_votes  ...       budget  \
6878          148.0           8.8       2779556  ...  160000000.0   

         recettes patrimonial zone_de_production annee_de_sortie  \
6878  825532764.0          No          Américain            2010   

                                            genres_list  \
6878  [Action, Aventure, Science-fiction, Science-fi...   

                                            synopsis_fr  \
6878  Cobb, un voleur habile qui commet de l'espionn...   

                                          keywords_list  \
6878  [mission, dreams, kidnapping, spy, allegory, i...   

                                         synopsis_clean  \
6878  co

In [37]:
import pandas as pd

df = pd.read_parquet("../data/processed/movies_with_keywords.parquet")
print(type(df["genres_list"].iloc[0]))
print(df["genres_list"].iloc[0])
print()
print(type(df["main_genre"].iloc[0]))
print(df["main_genre"].iloc[0])
print()
print(df.index[:10])
print(len(df))

<class 'numpy.ndarray'>
['Comédie' 'Fantastique' 'Romance']

<class 'str'>
Comédie

RangeIndex(start=0, stop=10, step=1)
10307


In [39]:
from scipy.sparse import load_npz
X = load_npz("../data/processed/X_final.npz")
print(X.shape)
print(len(df))

(10307, 11726)
10307
